# DSCI 503 — Comprehensive Homework Exercise
**Course:** UBA MADS — DSCI 503  
**Dataset:** NHANES (10,000 rows, 76 columns)  
**Total:** 500 pts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
## Part 1: Probability Foundations & Random Variables (40 pts)

### Q1 — Marginal Probabilities (5 pts)

**What:** Marginal probability is the simplest form — out of all respondents, what fraction has a given trait? Called 'marginal' because it lives in the margins of a joint frequency table.

**Key decision:** Denominator is `notna().sum()` (actual respondents), not `len(df)` (total rows). SmokeNow has 6,789 missing values — using `len(df)` would silently treat 'no answer' as 'No', dropping the true rate from 45.7% to 14.7%.

In [ ]:
# Load and select variables
df_raw = pd.read_csv('data/raw/nhanes.csv')
cols = ['Age', 'Gender', 'BMI', 'BPSysAve', 'Diabetes', 'PhysActive', 'SmokeNow', 'TotChol']
df = df_raw[cols].copy()

print('Dataset shape:', df.shape)
print()

# Marginal probabilities — denominator = respondents only (notna)
results = []
for col in ['Diabetes', 'PhysActive', 'SmokeNow']:
    n_yes = (df[col] == 'Yes').sum()
    n_respondents = df[col].notna().sum()
    n_missing = df[col].isna().sum()
    p = n_yes / n_respondents
    results.append({'Variable': col, 'P(Yes)': round(p, 4), 'Yes': n_yes,
                    'Respondents': n_respondents, 'Missing': n_missing})

summary = pd.DataFrame(results)
print(summary.to_string(index=False))

**Results:**
- P(Diabetes=Yes) = 0.0771 — ~7.7% of respondents have diabetes
- P(PhysActive=Yes) = 0.5584 — ~55.8% are physically active  
- P(SmokeNow=Yes) = 0.4566 — ~45.7% of *smokers who answered* currently smoke

**Why notna() matters:** SmokeNow has 6,789 missing. Using len(df)=10,000 gives 14.7% — a 31 percentage point error that misrepresents the smoking rate entirely.

### Q2 — Conditional Probabilities & Independence (5 pts)

**What:** P(A|B) = probability of A given we already know B is true. In code: filter to the B group, then compute proportion of A within that group.

**Independence test:** Two events are independent if P(A|B) = P(A). If knowing B changes the probability of A, they are dependent.

In [ ]:
# Q2 — Conditional Probabilities

# P(Diabetes=Yes | BMI > 30)
subset = df[df['BMI'] > 30]
p_diab_obese = (subset['Diabetes'] == 'Yes').sum() / subset['Diabetes'].notna().sum()

# P(Diabetes=Yes | BMI <= 30)
subset = df[df['BMI'] <= 30]
p_diab_not_obese = (subset['Diabetes'] == 'Yes').sum() / subset['Diabetes'].notna().sum()

# P(BPSysAve > 140 | Age > 60)
subset = df[df['Age'] > 60]
p_hbp_elderly = (subset['BPSysAve'] > 140).sum() / subset['BPSysAve'].notna().sum()

# P(PhysActive=Yes | Gender)
p_active_female = (df[df['Gender'] == 'female']['PhysActive'] == 'Yes').sum() / df[df['Gender'] == 'female']['PhysActive'].notna().sum()
p_active_male   = (df[df['Gender'] == 'male']['PhysActive'] == 'Yes').sum() / df[df['Gender'] == 'male']['PhysActive'].notna().sum()

# Independence test
p_diabetes = (df['Diabetes'] == 'Yes').sum() / df['Diabetes'].notna().sum()

print(f"P(Diabetes) overall:        {p_diabetes:.4f}")
print(f"P(Diabetes | BMI > 30):     {p_diab_obese:.4f}")
print(f"P(Diabetes | BMI <= 30):    {p_diab_not_obese:.4f}")
print(f"P(BP > 140 | Age > 60):     {p_hbp_elderly:.4f}")
print(f"P(PhysActive | female):     {p_active_female:.4f}")
print(f"P(PhysActive | male):       {p_active_male:.4f}")
print()
print(f"Independent (Diabetes, BMI>30)? {abs(p_diabetes - p_diab_obese) < 0.01}")

**Results:**
- P(Diabetes | BMI > 30) = 0.1557 vs P(Diabetes) = 0.0771 — obesity doubles diabetes risk
- P(Diabetes | BMI ≤ 30) = 0.0467 — non-obese have half the population average
- P(BP > 140 | Age > 60) = 0.2910 — 1 in 3 elderly adults has high blood pressure
- PhysActive: males (58.6%) slightly more active than females (53.2%)

**Independence:** P(Diabetes | BMI>30) = 0.1557 ≠ P(Diabetes) = 0.0771. They differ by a factor of 2, so Diabetes and obesity are **not independent** — knowing BMI changes the diabetes probability significantly.

### Q3 — Bayes' Theorem (5 pts)

**What:** Bayes' Theorem flips a conditional probability. We know P(Diabetes | Obese) but want P(Obese | Diabetes).

**Formula:** P(Obese | Diabetes) = P(Diabetes | Obese) × P(Obese) / P(Diabetes)

**Why not just compute directly?** In real life you often can't — e.g., P(Disease | Test Positive) requires observing all patients, but P(Test Positive | Disease) comes from controlled trials. Bayes lets you flip using pieces you already have.

In [ ]:
# Q3 — Bayes' Theorem: P(Obese | Diabetes)

# Step 1: compute pieces
p_obese = (df['BMI'] > 30).sum() / df['BMI'].notna().sum()
p_diabetes = (df['Diabetes'] == 'Yes').sum() / df['Diabetes'].notna().sum()

subset = df[df['BMI'] > 30]
p_diab_given_obese = (subset['Diabetes'] == 'Yes').sum() / subset['Diabetes'].notna().sum()

# Step 2: Bayes' Theorem
p_obese_given_diab = p_diab_given_obese * p_obese / p_diabetes

print("--- Bayes' Theorem ---")
print(f"P(Diabetes | Obese):    {p_diab_given_obese:.4f}")
print(f"P(Obese):               {p_obese:.4f}")
print(f"P(Diabetes):            {p_diabetes:.4f}")
print(f"P(Obese | Diabetes):    {p_obese_given_diab:.4f}  ← Bayes result")

# Step 3: direct verification
subset_diab = df[df['Diabetes'] == 'Yes']
p_direct = (subset_diab['BMI'] > 30).sum() / subset_diab['BMI'].notna().sum()
print(f"P(Obese | Diabetes):    {p_direct:.4f}  ← direct computation")
print(f"\nDifference: {abs(p_obese_given_diab - p_direct):.4f} (rounding in intermediate steps)")

**Result:** P(Obese | Diabetes) ≈ 0.57 — 57% of diabetics are obese.

**Key insight:** P(Diabetes | Obese) = 15.6% and P(Obese | Diabetes) = 57% are very different numbers. Bayes' Theorem is necessary when you can't directly observe the flipped condition — e.g., in medical testing where P(Disease | Positive Test) must be derived from P(Positive Test | Disease).

### Q4 — Visualization Dashboard (5 pts)

**What:** 4 subplots showing probability and distribution patterns in NHANES data.

1. **BMI histogram + KDE** — shape of the distribution; expect right skew (tail toward obesity)
2. **Age vs BPSysAve scatter + regression** — red line shows trend direction; positive slope = BP rises with age
3. **TotChol by Diabetes** — overlapping histograms show if diabetics have different cholesterol distribution
4. **Heatmap P(Diabetes | Age, BMI)** — darker = higher diabetes probability; expect bottom-right (old + obese) to be darkest

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1 — BMI histogram + KDE
ax1 = axes[0, 0]
ax1.hist(df['BMI'].dropna(), bins=40, density=True, alpha=0.6, color='steelblue')
df['BMI'].dropna().plot(kind='kde', ax=ax1, color='navy', linewidth=2)
ax1.set_xlabel('BMI')
ax1.set_ylabel('Density')
ax1.set_title('Distribution of BMI')

# 2 — Age vs BPSysAve scatter + regression line
ax2 = axes[0, 1]
mask = df['Age'].notna() & df['BPSysAve'].notna()
ax2.scatter(df.loc[mask, 'Age'], df.loc[mask, 'BPSysAve'], alpha=0.3, s=10, color='steelblue')
m, b = np.polyfit(df.loc[mask, 'Age'], df.loc[mask, 'BPSysAve'], 1)
x_line = np.linspace(df['Age'].min(), df['Age'].max(), 100)
ax2.plot(x_line, m * x_line + b, color='red', linewidth=2)
ax2.set_xlabel('Age')
ax2.set_ylabel('BPSysAve')
ax2.set_title('Age vs Blood Pressure')

# 3 — TotChol by Diabetes status
ax3 = axes[1, 0]
for label, color in [('Yes', 'tomato'), ('No', 'steelblue')]:
    subset = df[df['Diabetes'] == label]['TotChol'].dropna()
    ax3.hist(subset, bins=40, density=True, alpha=0.5, color=color, label=label)
ax3.set_xlabel('Total Cholesterol')
ax3.set_ylabel('Density')
ax3.set_title('TotChol by Diabetes Status')
ax3.legend(title='Diabetes')

# 4 — Heatmap: P(Diabetes | Age quartile, BMI quartile)
ax4 = axes[1, 1]
df2 = df.dropna(subset=['Age', 'BMI', 'Diabetes']).copy()
df2['age_group'] = pd.qcut(df2['Age'], 4, labels=['Q1','Q2','Q3','Q4'])
df2['bmi_group'] = pd.qcut(df2['BMI'], 4, labels=['Q1','Q2','Q3','Q4'])
heatmap_data = df2.groupby(['age_group', 'bmi_group'], observed=True).apply(
    lambda x: (x['Diabetes'] == 'Yes').sum() / len(x)
).unstack()
sns.heatmap(heatmap_data, ax=ax4, annot=True, fmt='.2f', cmap='YlOrRd')
ax4.set_xlabel('BMI Quartile')
ax4.set_ylabel('Age Quartile')
ax4.set_title('P(Diabetes | Age group, BMI group)')

plt.tight_layout()
plt.savefig('figures/q4_visualization_dashboard.png', dpi=150)
plt.show()

**Findings:**
- BMI is right-skewed — most people cluster around 25–30 but a long tail extends toward obesity
- Blood pressure increases with age (positive slope on regression line)
- Diabetics and non-diabetics have similar TotChol distributions with slight rightward shift for diabetics
- Heatmap confirms: diabetes probability is highest in Q4 age + Q4 BMI (oldest + most obese) — age and BMI together are stronger predictors than either alone

### Q5 — BMI as Continuous Random Variable, MLE & Normality Test (5 pts)

**What:** Treat BMI as a continuous RV. Fit a normal distribution via MLE, check if it actually fits.

**MLE for normal:** Maximizing the log-likelihood analytically gives μ_MLE = sample mean, σ_MLE = sample std. This is a mathematical result — `stats.norm.fit()` always returns the sample mean for μ.

**Normality checks:** Shapiro-Wilk tests H0: data is normal. p < 0.05 = reject normality. Q-Q plot shows deviation visually — points curving away from the diagonal = non-normal tails.

In [ ]:
from scipy import stats

bmi = df['BMI'].dropna()

# MLE fit — for normal, always returns sample mean and std
mu, sigma = stats.norm.fit(bmi)

print(f"MLE mu:    {mu:.4f}")
print(f"MLE sigma: {sigma:.4f}")
print(f"Mean:      {bmi.mean():.4f}")
print(f"Variance:  {bmi.var():.4f}")
print(f"Skewness:  {bmi.skew():.4f}")
print(f"Kurtosis:  {bmi.kurt():.4f}")

# Shapiro-Wilk (sample 5000 — full dataset too large for Shapiro-Wilk)
stat, p = stats.shapiro(bmi.sample(5000, random_state=42))
print(f"\nShapiro-Wilk stat: {stat:.4f}, p-value: {p:.6f}")
print("Normal fit: REJECTED (p < 0.05)")

# Plot: histogram + fitted normal + Q-Q plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
ax1.hist(bmi, bins=50, density=True, alpha=0.6, color='steelblue', label='Observed')
x = np.linspace(bmi.min(), bmi.max(), 200)
ax1.plot(x, stats.norm.pdf(x, mu, sigma), color='red', linewidth=2, label=f'Normal(μ={mu:.1f}, σ={sigma:.1f})')
ax1.set_xlabel('BMI')
ax1.set_ylabel('Density')
ax1.set_title('BMI: Observed vs Fitted Normal')
ax1.legend()

ax2 = axes[1]
stats.probplot(bmi, dist='norm', plot=ax2)
ax2.set_title('Q-Q Plot of BMI')

plt.tight_layout()
plt.savefig('figures/q5_bmi_normal_fit.png', dpi=150)
plt.show()

**Results:**
- MLE: μ = 26.66, σ = 7.38 — identical to sample mean/std (mathematical property of normal MLE)
- Skewness = 0.90 → right-skewed (obese tail pulls distribution right)
- Kurtosis = 2.20 → slightly flatter peak than normal (normal = 3.0)
- Shapiro-Wilk p ≈ 0 → reject normality — BMI is **not** normally distributed

**Conclusion:** Normal is a rough approximation for the middle of BMI but fails at the tails. The right skew from obesity violates the normality assumption.

### Q6 — DaysPhysHlthBad as Discrete RV, Poisson vs Negative Binomial (5 pts)

**What:** Count variables (whole numbers only) are modeled with Poisson, not normal. Poisson has one parameter λ = mean. Key assumption: mean = variance.

**Overdispersion:** When variance >> mean, Poisson fails. Negative Binomial adds a second parameter (r) that controls variance independently — it handles the spread Poisson can't.

**Chi-squared goodness-of-fit:** Compares observed counts vs Poisson-predicted counts. Large χ² = large gap = bad fit. H0: data follows Poisson. p < 0.05 = reject.

In [ ]:
from scipy.stats import nbinom

col = df_raw['DaysPhysHlthBad'].dropna().astype(int)
lam = col.mean()

print(f"λ (MLE) = {lam:.4f}")
print(f"Mean:     {col.mean():.4f}")
print(f"Variance: {col.var():.4f}")
print(f"Overdispersion ratio: {col.var()/col.mean():.2f}x")

# Observed vs Poisson expected frequencies
max_val = 30
observed = np.array([(col == i).sum() for i in range(max_val+1)])
poisson_probs = stats.poisson.pmf(range(max_val+1), lam)
poisson_expected = poisson_probs * len(col)

# Chi-squared test (only bins with expected >= 5)
mask = poisson_expected >= 5
chi2_stat = ((observed[mask] - poisson_expected[mask])**2 / poisson_expected[mask]).sum()
df_chi = mask.sum() - 2
p_poisson = 1 - stats.chi2.cdf(chi2_stat, df_chi)
print(f"\nPoisson chi-squared: stat={chi2_stat:.2f}, p={p_poisson:.6f}")
print(f"Poisson fit: {'ACCEPTED' if p_poisson > 0.05 else 'REJECTED'}")

# Negative Binomial fit via method of moments
mean = col.mean()
var = col.var()
p_nb = mean / var
r_nb = mean**2 / (var - mean)
print(f"\nNegative Binomial: r={r_nb:.4f}, p={p_nb:.4f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(0, 31)

ax1 = axes[0]
ax1.bar(x - 0.2, observed/len(col), width=0.4, label='Observed', color='steelblue', alpha=0.7)
ax1.bar(x + 0.2, poisson_probs, width=0.4, label='Poisson', color='tomato', alpha=0.7)
ax1.set_xlabel('Days of Bad Health'); ax1.set_ylabel('Probability')
ax1.set_title('Observed vs Poisson PMF'); ax1.legend(); ax1.set_xlim(-1, 20)

nb_probs = nbinom.pmf(x, r_nb, p_nb)
ax2 = axes[1]
ax2.bar(x - 0.2, observed/len(col), width=0.4, label='Observed', color='steelblue', alpha=0.7)
ax2.bar(x + 0.2, nb_probs, width=0.4, label='Neg Binomial', color='green', alpha=0.7)
ax2.set_xlabel('Days of Bad Health'); ax2.set_ylabel('Probability')
ax2.set_title('Observed vs Negative Binomial PMF'); ax2.legend(); ax2.set_xlim(-1, 20)

plt.tight_layout()
plt.savefig('figures/q6_poisson_nb_fit.png', dpi=150)
plt.show()

**Results:**
- Overdispersion ratio = 16.42x — variance is 16x the mean, violating Poisson's mean=variance assumption
- Poisson chi-squared p ≈ 0 → REJECTED — Poisson cannot fit this data
- Negative Binomial (r=0.22, p=0.06) handles the overdispersion — visually fits the observed distribution much better

**Conclusion:** DaysPhysHlthBad is overdispersed count data. Poisson fails because it assumes mean=variance. Negative Binomial adds a second parameter controlling variance independently, making it the appropriate model here.

### Q7 — E[X] and Var[X]: Analytical vs Theoretical (5 pts)

**What:** Compare sample E[X] and Var[X] against what the fitted distribution predicts. A good model's theoretical moments match the data.

**Key insight:** 
- Normal MLE sets μ = sample mean, σ² ≈ sample variance → discrepancy is ~0 by construction
- Poisson forces Var = λ = mean → when data is overdispersed, theoretical Var is wildly wrong
- Negative Binomial fitted via Method of Moments → sets parameters so theoretical moments equal sample moments exactly → discrepancy = 0 by construction

In [ ]:
# Q7 — E[X] and Var[X]: sample vs theoretical

# BMI — Normal
bmi = df['BMI'].dropna()
mu, sigma = stats.norm.fit(bmi)

print('=== BMI — Normal Distribution ===')
print(f'  Sample E[X]:      {bmi.mean():.4f}')
print(f'  Theoretical E[X]: {mu:.4f}')
print(f'  Sample Var[X]:    {bmi.var():.4f}')
print(f'  Theoretical Var:  {sigma**2:.4f}')
print(f'  Discrepancy E[X]: {abs(bmi.mean() - mu):.6f}')
print(f'  Discrepancy Var:  {abs(bmi.var() - sigma**2):.6f}')

# DaysPhysHlthBad — Poisson
col = df_raw['DaysPhysHlthBad'].dropna().astype(int)
lam = col.mean()
mean = col.mean()
var = col.var()
p_nb = mean / var
r_nb = mean**2 / (var - mean)

print()
print('=== DaysPhysHlthBad — Poisson ===')
print(f'  Sample E[X]:      {col.mean():.4f}')
print(f'  Theoretical E[X]: {lam:.4f}')
print(f'  Sample Var[X]:    {col.var():.4f}')
print(f'  Theoretical Var:  {lam:.4f}  (Poisson: Var = lambda)')
print(f'  Discrepancy Var:  {abs(col.var() - lam):.4f}  <- confirms bad fit')

# DaysPhysHlthBad — Negative Binomial
nb_mean = r_nb * (1 - p_nb) / p_nb
nb_var  = r_nb * (1 - p_nb) / p_nb**2
print()
print('=== DaysPhysHlthBad — Negative Binomial ===')
print(f'  Sample E[X]:      {col.mean():.4f}')
print(f'  Theoretical E[X]: {nb_mean:.4f}')
print(f'  Sample Var[X]:    {col.var():.4f}')
print(f'  Theoretical Var:  {nb_var:.4f}')
print(f'  Discrepancy E[X]: {abs(col.mean() - nb_mean):.6f}')
print(f'  Discrepancy Var:  {abs(col.var() - nb_var):.6f}')

**Results:**
- Normal (BMI): E[X] discrepancy = 0, Var discrepancy ≈ 0 — perfect match by MLE construction
- Poisson (count): E[X] matches but Var discrepancy = 51.43 — Poisson cannot model overdispersion
- Negative Binomial: both discrepancies = 0 — MOM fitting sets parameters to match sample moments exactly

**Conclusion:** Model fit quality shows up in moment discrepancies. Poisson's theoretical variance (3.33) vs sample variance (54.77) quantifies exactly why it fails. Negative Binomial resolves this by decoupling mean and variance.

### Q8 — 2D KDE of BMI + BPSysAve, Correlation & Independence (5 pts)

**What:** 2D KDE estimates a smooth joint density surface for two continuous variables simultaneously. Contours show where most people cluster. Marginals on each axis show individual distributions.

**Independence test:** Two variables are independent if knowing one tells you nothing about the other. Correlation ≠ 0 means they are dependent. We also look at the shape of the joint distribution — an oval tilted diagonally indicates correlation.

In [ ]:
clean = df[['BMI', 'BPSysAve']].dropna()

# Correlation and covariance
corr = clean['BMI'].corr(clean['BPSysAve'])
cov  = clean['BMI'].cov(clean['BPSysAve'])
print(f"Correlation: {corr:.4f}")
print(f"Covariance:  {cov:.4f}")
print(f"Independent: {abs(corr) < 0.05}")

# 2D KDE with marginals
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 2, width_ratios=[4,1], height_ratios=[1,4], hspace=0.05, wspace=0.05)

ax_main  = fig.add_subplot(gs[1, 0])
ax_top   = fig.add_subplot(gs[0, 0], sharex=ax_main)
ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)

sns.kdeplot(x=clean['BMI'], y=clean['BPSysAve'], ax=ax_main, fill=True, cmap='Blues', levels=10)
ax_main.set_xlabel('BMI')
ax_main.set_ylabel('BPSysAve')
ax_main.set_title(f'Joint Density — Correlation={corr:.3f}')

ax_top.hist(clean['BMI'], bins=50, density=True, color='steelblue', alpha=0.7)
ax_top.set_ylabel('Density')
plt.setp(ax_top.get_xticklabels(), visible=False)

ax_right.hist(clean['BPSysAve'], bins=50, density=True, color='steelblue', alpha=0.7, orientation='horizontal')
ax_right.set_xlabel('Density')
plt.setp(ax_right.get_yticklabels(), visible=False)

plt.savefig('figures/q8_2d_kde.png', dpi=150, bbox_inches='tight')
plt.show()

**Results:**
- Correlation = 0.2617 — weak positive relationship: higher BMI tends toward higher BP
- Covariance = 31.51 — positive, confirming same direction
- Independent: False — correlation ≠ 0, so BMI and BPSysAve are not independent

**Conclusion:** The joint density contours tilt diagonally (not circular), confirming weak but real dependence. Higher BMI is associated with higher blood pressure, consistent with clinical knowledge. However, correlation = 0.26 means BMI alone explains only ~7% of BP variance (r²).